Загрузка и обработка датасета `irlspbru/RusLawOD`. Итоговый скрипт для загрузки в `src/data/load.py`

In [ ]:
from datasets import load_dataset
from datetime import date

In [2]:
dataset = load_dataset("irlspbru/RusLawOD", data_files=["ruslawod_01.parquet"])

In [3]:
dataset = dataset.filter(lambda x: x["textIPS"] is not None).map(lambda x: {"word_count": len(x["textIPS"].split())})

In [4]:
dataset = dataset.remove_columns(["pravogovruNd", "actual_datetimeIPS", "actual_datetime_humanIPS", "taggedtextIPS", "keywordsByIPS", "classifierByIPS", "is_widely_used", "statusIPS"])

In [5]:
dataset = dataset.filter(lambda x: x["word_count"] < 50 or x["word_count"] > 2000)

In [6]:
df = dataset["train"].to_polars()

In [7]:
df.group_by("doc_typeIPS").len().sort("len", descending=True)

doc_typeIPS,len
str,u32
"""Приказ""",1759
"""Распоряжение""",1246
"""Постановление""",1000
"""Указ""",788
"""Федеральный закон""",216
…,…
"""Циркулярное письмо""",3
"""Протокол""",2
"""Основы законодательства""",1


In [8]:
keep_doc_types = set(df.group_by("doc_typeIPS").len().sort("len", descending=True)[:5]["doc_typeIPS"])

In [9]:
dataset = dataset.filter(lambda x: x["doc_typeIPS"] in keep_doc_types)

Filter:   0%|          | 0/5148 [00:00<?, ? examples/s]

In [10]:
dataset = dataset.class_encode_column("doc_typeIPS")

Flattening the indices:   0%|          | 0/5009 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/5009 [00:00<?, ? examples/s]

In [12]:
dataset = dataset["train"].train_test_split(test_size=100, train_size=200, stratify_by_column="doc_typeIPS")

In [13]:
dataset = dataset.map(lambda x: {"doc_type": dataset["train"].features["doc_typeIPS"].int2str(x["doc_typeIPS"])}).remove_columns(["doc_typeIPS"])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [15]:
dataset = dataset.rename_columns({
    "issuedByIPS": "act_full_name",
    "docdateIPS": "document_publication_date",
    "docNumberIPS": "document_number",
    "headingIPS": "title",
    "doc_author_normal_formIPS": "government_agency_name",
    "signedIPS": "signatory",
    "doc_type": "act_type",
    "textIPS": "act_text",
    }).remove_columns(["word_count"])

In [ ]:
def parse_date(x):
    date_str = x["publication_date"]
    day, month, year = date_str.split(".")
    return {"publication_date": date(int(year), int(month), int(day))}
dataset = dataset.map(parse_date)

In [16]:
dataset.save_to_disk("../data/raw/")

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]